# Text Processing

Language is continous, Models are discrete. Preprocessing is the bridge.

Tokenization, Stemming, Lemmatization

## Problem Definition

A model cannot read "The cats were running.", It reads integers.

* Where does a word start.
* What is the root of the word.
* How do we treat "run", "runing", "ran" as the samme thing when it helps, and as different things when it doesn't.

## Basic Concept

* Tokenization splits a string into tokens.
    * Word level for classical NLP
    * Subword for transformers.
* Stemming chops suffixes with ruls.
    * "running" --> "run"
    * "organization" --> "organ"
* Lemmatization reduces a word to its dictionary form using grammer knowledge.
    * Needs to know "ran" is past tense of "run"
    * Needs to know comparative forms "better" to good

# Build your Own

## Regex word tokenizer

Simplest useful tokenizer: splits on non-alphanumeric characters.

In [1]:
import re

def tokenize(text):
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\sA-Za-z0-9]", text)

tokenize("The cats weren't running at 3pm.")

['The', 'cats', "weren't", 'running', 'at', '3', 'pm', '.']

## Porter stemmer

The full porter algorithm has five phases of rules. Here is Step 1a alone.

In [4]:
def stem_step_1a(word):
    if word.endswith("sses"):
        return word[:-2]
    elif word.endswith("ies"):
        return word[:-2]
    elif word.endswith("ss"):
        return word
    elif word.endswith("s") and len(word) > 1:
        return word[:-1]
    else:
        return word

# ponies -> poni  step 1b would fix it.
[stem_step_1a(w) for w in ["caresses", "ponies", "caress", "cats"]]

['caress', 'poni', 'caress', 'cat']

## Lookup-based lemmatizer

In [5]:
LEMMA_TABLE = {
    ("running", "VERB"): "run",
    ("ran", "VERB"): "run",
    ("runs", "VERB"): "run",
    ("better", "ADJ"): "good",
    ("best", "ADJ"): "good",
    ("cats", "NOUN"): "cat",
    ("cat", "NOUN"): "cat",
    ("were", "VERB"): "be",
    ("was", "VERB"): "be",
    ("is", "VERB"): "be",
}

def lemmatize(word, pos):
    key = (word.lower(), pos)
    if key in LEMMA_TABLE:
        return LEMMA_TABLE[key]
    if pos == "VERB" and  word.endswith("ing"):
        return word[:-3]
    if pos == "NOUN" and word.endswith("s"):
        return word[:-1]
    return word.lower()

[lemmatize(w, p) for w, p in [("running", "VERB"), ("cats", "NOUN"), ("better", "ADJ")]]



['run', 'cat', 'good']

## Pipe them together

In [ ]:
def preprocess(text, pos_tagger=None):
    tokens = tokenize(text)
    stems = [stem_step_1a(t) for t in tokens]
    tags = pos_tagger(tokens) if pos_tagger else [(t, "NOUN") for t in tokens]
    lemmas = [lemmatize(w, p) for w, p in tags]
    return {
        "tokens": tokens,
        "stems": stems,
        "lemmas": lemmas,
    }

preprocess("The cats were running at 3pm.")